In [1]:
!pip install gradio

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os, zipfile, io, time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, GRU, Dense, Dropout, Reshape
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

WINDOW_SIZE  = 5
STEP_SIZE    = 2
FEATURE_COLS = ['HeartRate', 'Steps', 'Calories', 'Intensity', 'MET_actual']
N_FEATURES   = len(FEATURE_COLS)
N_CLASSES    = 3
CLASS_NAMES  = ['Baseline', 'Active', 'Fatigued']
CLASS_COLORS = ['#4CAF50', '#2196F3', '#F44336']

In [3]:

def build_cnn_gru(input_shape, n_classes=3):
    """
    CNN-GRU Hybrid Model for multivariate time-series classification.

    Parameters
    ----------
    input_shape : tuple (H, W, C) e.g. (1, WINDOW_SIZE, N_FEATURES)
    n_classes   : number of output classes

    Returns
    -------
    model : compiled Keras Model
    """
    inp = Input(shape=input_shape, name='Input_Window')

    x = Conv2D(32,  (3, 3), activation='relu', padding='same', name='Conv2D_32')(inp)
    x = MaxPooling2D(pool_size=(1, 2), name='MaxPool_1')(x)

    x = Conv2D(64,  (3, 3), activation='relu', padding='same', name='Conv2D_64')(x)
    x = MaxPooling2D(pool_size=(1, 2), name='MaxPool_2')(x)

    x = Conv2D(128, (3, 3), activation='relu', padding='same', name='Conv2D_128')(x)
    x = BatchNormalization(name='BatchNorm')(x)

    x = Flatten(name='Flatten')(x)
    flat_dim = int(x.shape[-1])
    x = Reshape((1, flat_dim), name='Reshape_for_GRU')(x)

    x = GRU(128, return_sequences=True,  name='GRU_128')(x)
    x = GRU(64,  return_sequences=False, name='GRU_64')(x)


    x   = Dense(512, activation='relu', name='Dense_512')(x)
    x   = Dropout(0.3, name='Dropout_0.3')(x)
    out = Dense(n_classes, activation='softmax', name='Output_Softmax')(x)

    return Model(inputs=inp, outputs=out, name='CNN_GRU_Fatigue_Classifier')


INPUT_SHAPE = (1, WINDOW_SIZE, N_FEATURES)
model = build_cnn_gru(INPUT_SHAPE, N_CLASSES)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

total_params = model.count_params()
print(f'\n  Total parameters : {total_params:,}')
print(f'  Trainable params : {sum(np.prod(v.shape) for v in model.trainable_variables):,}')

Model: "CNN_GRU_Fatigue_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Window (InputLayer)       │ (None, 1, 5, 5)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_32 (Conv2D)              │ (None, 1, 5, 32)       │         1,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool_1 (MaxPooling2D)        │ (None, 1, 2, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_64 (Conv2D)              │ (None, 1, 2, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool_2 (MaxPooling2D)        │ (None, 1, 1, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_128 (Conv2D)             │ (None, 1, 1, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ BatchNorm (BatchNormalization)  │ (None, 1, 1, 128)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Reshape_for_GRU (Reshape)       │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ GRU_128 (GRU)                   │ (None, 1, 128)         │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ GRU_64 (GRU)                    │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_512 (Dense)               │ (None, 512)            │        33,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_0.3 (Dropout)           │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Softmax (Dense)          │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 265,475 (1.01 MB)

 Trainable params: 265,219 (1.01 MB)

 Non-trainable params: 256 (1.00 KB)


  Total parameters : 265,475
  Trainable params : 265,219


In [5]:
import gradio as gr
import numpy as np
import pandas as pd
import tensorflow as tf

input_shape = (1, 5, 5)
model = build_cnn_gru(input_shape)

model.load_weights('/content/model.weights.h5')

WINDOW_SIZE = 5
N_FEATURES = 5
FEATURE_COLS = ['HeartRate', 'Steps', 'Calories', 'Intensity', 'MET_actual']
CLASS_NAMES = ['Baseline', 'Active', 'Fatigued']


POP_MEANS = np.array([78.0, 50.0, 3.8, 0.9, 2.8])
POP_STDS  = np.array([18.0, 65.0, 2.8, 1.0, 2.5])

def predict_state(df):
    if model is None:
        return {"Error: Model failed to load": 1.0}, "No advice"

    try:
        raw_data = df[FEATURE_COLS].values.astype(np.float32)
        if raw_data.shape != (WINDOW_SIZE, N_FEATURES):
            return {"Error: Input must be exactly 5 minutes": 1.0}, ""


        norm_w = (raw_data - POP_MEANS) / POP_STDS


        model_input = norm_w.reshape(1, 1, WINDOW_SIZE, N_FEATURES)


        probs = model.predict(model_input, verbose=0)[0]
        result_dict = {CLASS_NAMES[i]: float(probs[i]) for i in range(len(CLASS_NAMES))}

        pred_class = CLASS_NAMES[np.argmax(probs)]
        if pred_class == "Fatigued":
            advice = "User exhibits extreme fatigue/vigorous stress. Rest advised."
        elif pred_class == "Active":
            advice = "User is in an active/exercise state."
        else:
            advice = "User is in a normal resting baseline."

        return result_dict, advice

    except Exception as e:
        return {f"Error: {str(e)}": 1.0}, ""



baseline_scenario = pd.DataFrame([
    [65, 0, 1.0, 0, 1.0],
    [67, 0, 1.1, 0, 1.0],
    [64, 0, 1.0, 0, 1.0],
    [65, 0, 1.0, 0, 1.0],
    [66, 0, 1.1, 0, 1.0]
], columns=FEATURE_COLS)


active_scenario = pd.DataFrame([
    [95,  85, 4.5, 1, 3.5],
    [98,  90, 4.8, 1, 3.8],
    [102, 92, 5.0, 2, 4.0],
    [100, 88, 4.9, 1, 3.9],
    [105, 95, 5.2, 2, 4.1]
], columns=FEATURE_COLS)

fatigue_scenario = pd.DataFrame([
    [155, 0, 9.0, 3, 7.8],
    [160, 0, 9.5, 3, 8.2],
    [158, 0, 9.2, 3, 8.0],
    [162, 0, 9.8, 3, 8.5],
    [159, 0, 9.6, 3, 8.3]
], columns=FEATURE_COLS)


In [6]:

custom_theme = gr.themes.Soft(primary_hue="indigo", secondary_hue="blue")

with gr.Blocks(theme=custom_theme, title="Fatigue Prediction Demo") as demo:

    gr.Markdown("# Wearable Fatigue Prediction System")
    gr.Markdown("**CNN-GRU Architecture predicting physiological states from 5-minute Fitbit data windows.**")
    gr.Markdown("---")

    with gr.Row():

        with gr.Column(scale=5):
            gr.Markdown("### Input Vitals (5 Minutes)")

            input_df = gr.Dataframe(
                headers=FEATURE_COLS,
                row_count=(5, "fixed"),
                col_count=(5, "fixed"),
                value=baseline_scenario,
                interactive=True,
                label="Sample data"
            )

            predict_btn = gr.Button("Analyze Window", variant="primary")

            gr.Markdown("### Quick-Load Scenarios")
            with gr.Row():
                btn_base = gr.Button("Desk / Resting", size="sm")
                btn_act = gr.Button("Brisk Walk", size="sm")
                btn_fat = gr.Button("Vigorous / Fatigued", size="sm")


        with gr.Column(scale=4):
            gr.Markdown("### Analysis Results")

            output_label = gr.Label(num_top_classes=3, label="Predicted State")
            advice_box = gr.Textbox(label="System Recommendation", lines=2)


    btn_base.click(fn=lambda: baseline_scenario, outputs=input_df)
    btn_act.click(fn=lambda: active_scenario, outputs=input_df)
    btn_fat.click(fn=lambda: fatigue_scenario, outputs=input_df)
    predict_btn.click(fn=predict_state, inputs=input_df, outputs=[output_label, advice_box])

if __name__ == "__main__":
    demo.launch(share=True)


/tmp/ipykernel_10384/2202709636.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, title="Fatigue Prediction Demo") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f5068fd69493caece.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
